In [5]:
# ============================================================
# KENYA WATER QUALITY — CLEAN & LOAD PIPELINE (SINGLE CELL)
# ============================================================
# Update the MySQL credentials below, then run this cell.
# ============================================================

import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# ---------- STEP 0: Set working directory ----------
PROJECT_DIR = r"C:\Users\user\Desktop\Final Project"
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

# ---------- STEP 1: MySQL credentials ----------
MYSQL_USER     = "root"
MYSQL_PASSWORD = "Kellyshi244"
MYSQL_HOST     = "localhost"
MYSQL_PORT     = 3306
MYSQL_DATABASE = "kenya_water"

# ---------- Helper functions ----------
def header(t):
    print("\n" + "=" * 60)
    print(t)
    print("=" * 60)

def audit_nulls(df, name):
    print(f"\nNull counts in {name}:")
    found = False
    for col, count in df.isnull().sum().items():
        if count > 0:
            print(f"   {col}: {count} nulls ({count/len(df)*100:.1f}%)")
            found = True
    if not found:
        print("   ✅ No nulls found")

def clean_columns(df):
    df.columns = [
        str(c).strip().replace(" ", "_").replace("%", "pct")
        .replace("(", "").replace(")", "").replace("/", "_").replace("-", "_")
        for c in df.columns
    ]
    return df

# ============================================================
# PART 1: CLEAN NATIONAL INDICATORS
# ============================================================
header("PART 1: CLEANING NATIONAL INDICATORS")

raw = pd.read_csv("data/processed/kenya_water_master.csv")
print(f"Loaded {len(raw)} rows")

audit = raw.groupby(["source", "indicator_code", "indicator_name"]).agg(
    n_years=("year", "nunique"),
    first_year=("year", "min"),
    last_year=("year", "max"),
).reset_index()
print("\nCoverage audit:")
print(audit.to_string(index=False))

MIN_YEARS = 15
keep = audit[audit["n_years"] >= MIN_YEARS]["indicator_code"].tolist()
drop = audit[audit["n_years"] < MIN_YEARS]["indicator_code"].tolist()
print(f"\n✅ Keeping {len(keep)}: {keep}")
print(f"❌ Dropping {len(drop)}: {drop}")

national_clean = raw[raw["source"] == "world_bank"].copy()
national_clean = national_clean[national_clean["indicator_code"].isin(keep)]
national_clean = national_clean.dropna(subset=["value"])
national_clean["value"] = national_clean["value"].round(2)
national_clean["year"] = national_clean["year"].astype(int)
national_clean = national_clean.sort_values(["indicator_name", "year"]).reset_index(drop=True)
national_clean.to_csv("data/processed/kenya_water_clean.csv", index=False)
audit.to_csv("data/processed/coverage_audit.csv", index=False)
print(f"\n✅ Saved {len(national_clean)} rows → data/processed/kenya_water_clean.csv")

# ============================================================
# PART 2: CLEAN ATHI RIVER
# ============================================================
header("PART 2: CLEANING ATHI RIVER")

athi = None
try:
    athi = pd.read_csv("data/raw/fixed/athi_river_coverage.csv")
    print(f"Loaded {len(athi)} rows, {len(athi.columns)} columns")
    print(f"Columns: {list(athi.columns)}")
    audit_nulls(athi, "Athi River")

    athi = clean_columns(athi)
    critical = [c for c in ["Type_of_Coverage", "Served_Pop"] if c in athi.columns]
    if critical:
        before = len(athi)
        athi = athi.dropna(subset=critical)
        print(f"\nDropped {before - len(athi)} rows missing critical fields")

    for col in athi.columns:
        if any(k in col.lower() for k in ["pop", "pct", "coverage", "growth"]):
            athi[col] = pd.to_numeric(athi[col], errors="coerce")

    athi.to_csv("data/processed/athi_clean.csv", index=False)
    print(f"\n✅ Saved {len(athi)} rows → data/processed/athi_clean.csv")
except FileNotFoundError:
    print("⚠️ Athi file not found — skipping")
except Exception as e:
    print(f"❌ Athi error: {e}")

# ============================================================
# PART 3: INSPECT & CLEAN KEWI
# ============================================================
header("PART 3: INSPECTING & CLEANING KEWI")

kewi = None
try:
    kewi = pd.read_csv("data/raw/fixed/kewi_water_tests.csv")
    print(f"Loaded {len(kewi)} rows, {len(kewi.columns)} columns")
    print(f"Columns: {list(kewi.columns)}")
    audit_nulls(kewi, "KEWI")

    print(f"\nColumn value diversity:")
    for col in kewi.columns:
        n_unique = kewi[col].nunique()
        samples = list(kewi[col].dropna().unique()[:5])
        print(f"   {col}: {n_unique} unique | samples: {samples}")

    kewi = clean_columns(kewi)

    before = len(kewi)
    kewi = kewi.dropna(how="all")
    print(f"\nDropped {before - len(kewi)} fully-empty rows")

    result_cols = [
        c for c in kewi.columns
        if any(k in c.lower() for k in ["result", "value", "measurement", "concentration"])
    ]
    if result_cols:
        before = len(kewi)
        kewi = kewi.dropna(subset=result_cols[:1])
        print(f"Dropped {before - len(kewi)} rows missing result values (column: {result_cols[0]})")
    else:
        print("⚠️ No 'result' column identified — no rows dropped")

    for col in kewi.select_dtypes(include="object").columns:
        kewi[col] = kewi[col].astype(str).str.strip()

    kewi.to_csv("data/processed/kewi_clean.csv", index=False)
    print(f"\n✅ Saved {len(kewi)} rows → data/processed/kewi_clean.csv")
except FileNotFoundError:
    print("⚠️ KEWI file not found — skipping")
except Exception as e:
    print(f"❌ KEWI error: {e}")

# ============================================================
# PART 4: RELOAD ALL THREE INTO MYSQL
# ============================================================
header("PART 4: RELOADING MYSQL")

base_url = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/"

try:
    base_engine = create_engine(base_url)
    with base_engine.connect() as conn:
        conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}"))
        conn.commit()
    print(f"✅ Database '{MYSQL_DATABASE}' ready")
except Exception as e:
    print(f"❌ MySQL connection failed: {e}")
    print("   Check credentials at the top of this cell.")
    raise

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

tables = {
    "national_indicators": "data/processed/kenya_water_clean.csv",
    "athi_river_coverage": "data/processed/athi_clean.csv",
    "kewi_water_tests":    "data/processed/kewi_clean.csv",
}

for name, path in tables.items():
    if not os.path.exists(path):
        print(f"⚠️  {name}: {path} not found — skipping")
        continue
    try:
        df = pd.read_csv(path)
        df.to_sql(name, engine, if_exists="replace", index=False)
        print(f"✅ {name}: {len(df)} rows loaded")
    except Exception as e:
        print(f"❌ {name}: {e}")

# ============================================================
# PART 5: VERIFY
# ============================================================
header("VERIFICATION")

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    for row in result:
        table = row[0]
        count = conn.execute(text(f"SELECT COUNT(*) FROM `{table}`")).fetchone()[0]
        print(f"   {table}: {count} rows")

print("\n✅ DONE")

Working directory: c:\Users\user\Desktop\Final Project

PART 1: CLEANING NATIONAL INDICATORS
Loaded 174 rows

Coverage audit:
    source  indicator_code                                  indicator_name  n_years  first_year  last_year
   who_gho WSH_WATER_BASIC                            Basic drinking water       25        2000       2024
world_bank  ER.H2O.FWTL.ZS Freshwater withdrawal (% of internal resources)       23        2000       2022
world_bank  SH.H2O.BASW.ZS           Basic drinking water services (% pop)       25        2000       2024
world_bank  SH.STA.BASS.ZS               Basic sanitation services (% pop)       25        2000       2024
world_bank  SH.STA.ODFC.ZS                         Open defecation (% pop)       25        2000       2024
world_bank  SH.STA.WASH.P5           Mortality from unsafe WASH (per 100k)        1        2019       2019

✅ Keeping 5: ['WSH_WATER_BASIC', 'ER.H2O.FWTL.ZS', 'SH.H2O.BASW.ZS', 'SH.STA.BASS.ZS', 'SH.STA.ODFC.ZS']
❌ Dropping 1: ['SH.

C:\Users\user\AppData\Local\Temp\ipykernel_27744\989197811.py:145: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in kewi.select_dtypes(include="object").columns:


OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)